# Preparación y limpieza de los datos de exportaciones peruanas, 2011–2025

Este cuaderno integra los tres archivos Excel de MINCETUR, selecciona los registros de exportación de 2011 a 2025 y unifica la clasificación sectorial tomando 2025 como referencia.

Para ejecutarlo, coloca los tres archivos Excel en `datos/originales/` y ejecuta las celdas en orden.

La base depurada se guarda en `datos/procesados/exportaciones_2011_2025_base_depurada.parquet`. Este archivo es la entrada del cuaderno `02_homologacion_1a1.ipynb`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# 1. Configuración de rutas, archivos y estructura de datos.
CARPETA_ORIGEN = Path("datos/originales")
CARPETA_SALIDA = Path("resultados/limpieza")
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

ARCHIVOS = [
    "BD XM-2011-2015.xlsx",
    "BD XM-2016-2020.xlsx",
    "BD XM-2021-2026_Agosto.xlsx",
]

COLUMNAS = [
    "flujo_comercial", "Año", "Mes", "Subpartida", "Descripción", "Sector",
    "Tipo", "codigo_pais", "Destino/Origen", "Continente", "Bloque 1",
    "Bloque 2", "US$ millones", "TM", "CUODE", "codigo_via",
    "Vía de Transporte",
]
CAMBIOS_ENCABEZADOS = {
    "Via de Transporte": "Vía de Transporte",
    "Millones": "US$ millones",
}
EXPORTACION = {"EXPORTACION", "EXPORTACIONES", "EXPO", "EXP", "X"}
IMPORTACION = {"IMPORTACION", "IMPORTACIONES", "IMPO", "IMP", "M", "I"}

# Comprobación de los archivos de entrada.
for nombre in ARCHIVOS:
    if not (CARPETA_ORIGEN / nombre).is_file():
        raise FileNotFoundError(f"Falta el archivo: {CARPETA_ORIGEN / nombre}")

In [ ]:
# 2. Revisión de encabezados de las hojas disponibles.
# Se incluye 2026 para verificar la estructura, pero no forma parte de la consolidación.
encabezados = []
referencia_encabezados = None

for nombre in ARCHIVOS:
    with pd.ExcelFile(CARPETA_ORIGEN / nombre) as libro:
        for hoja in libro.sheet_names:
            originales = list(pd.read_excel(libro, sheet_name=hoja, nrows=0).columns)
            if referencia_encabezados is None:
                referencia_encabezados = set(originales)
            encabezados.append({
                "Archivo": nombre, "Hoja": hoja, "Cantidad": len(originales),
                "Faltantes respecto a 2011": "; ".join(sorted(referencia_encabezados - set(originales))),
                "Adicionales respecto a 2011": "; ".join(sorted(set(originales) - referencia_encabezados)),
            })

# Exportación del resultado de la revisión.
pd.DataFrame(encabezados).to_csv(
    CARPETA_SALIDA / "encabezados_revisados.csv", index=False, encoding="utf-8-sig"
)

assert len(encabezados) == 16, "Revisar la cantidad de hojas disponibles."

In [ ]:
# 3. Lectura y preparación de las bases de exportaciones.
bases = []
control = []
for nombre in ARCHIVOS:
    with pd.ExcelFile(CARPETA_ORIGEN / nombre) as libro:
        for hoja in libro.sheet_names:
            if not str(hoja).isdigit():
                raise ValueError(f"Hoja con nombre inesperado: {hoja}")
            anio = int(hoja)
            if not 2011 <= anio <= 2025:
                continue  # Se excluyen los registros del 2026.

            # Lectura de datos y tratamiento de valores nulos.
            # Se conserva 'NA' como código de país de Namibia.
            tabla = pd.read_excel(
                libro, sheet_name=hoja,
                dtype={"Subpartida": "string", "codigo_pais": "string",
                       "codigo_via": "string"},
                keep_default_na=False, na_values=[""],
            )
            
            vacias = int(tabla.isna().all(axis=1).sum())
            tabla = tabla.dropna(how="all").copy()
            tabla.columns = tabla.columns.astype(str).str.strip()
            tabla = tabla.rename(columns=CAMBIOS_ENCABEZADOS)
            
            if tabla.columns.duplicated().any() or set(tabla.columns) != set(COLUMNAS):
                raise ValueError(f"Encabezados inesperados en {nombre}, hoja {hoja}")
            tabla = tabla[COLUMNAS]

            # Estandarización del flujo comercial y selección de exportaciones.
            flujo = (tabla["flujo_comercial"].astype("string").str.strip()
             .str.normalize("NFKD")
             .str.replace("[\u0300-\u036f]", "", regex=True)
             .str.upper())
            if not flujo.isin(EXPORTACION | IMPORTACION).all():
                raise ValueError(f"Flujo comercial no reconocido en {hoja}")
            total_export = int(flujo.isin(EXPORTACION).sum())
            tabla = tabla.loc[flujo.isin(EXPORTACION)].copy()

            # Validación del periodo y normalización de las subpartidas.
            tabla["Año"] = pd.to_numeric(tabla["Año"], errors="raise")
            tabla["Mes"] = pd.to_numeric(tabla["Mes"], errors="raise")
            if not tabla["Año"].eq(anio).all() or not tabla["Mes"].between(1, 12).all():
                raise ValueError(f"Año o mes inconsistente en hoja {hoja}")

            codigos = (tabla["Subpartida"].astype("string").str.strip()
                       .str.replace(r"\.0$", "", regex=True))
            if not codigos.str.fullmatch(r"\d{1,10}").fillna(False).all():
                raise ValueError(f"Códigos de subpartida con formato inválido en {hoja}")
            tabla["Subpartida"] = codigos.str.zfill(10)
            
            # Conversión de las variables numéricas.
            for campo in ["US$ millones", "TM"]:
                tabla[campo] = pd.to_numeric(tabla[campo], errors="raise")

            bases.append(tabla)
            control.append({"Año": anio, "Filas vacías": vacias,
                            "Exportaciones": total_export})

# Consolidación de las exportaciones del periodo 2011–2025.
df = pd.concat(bases, ignore_index=True)

# Comprobaciones de integridad y tratamiento del código de Namibia.
assert len(control) == 15 and len(df) == 1_988_951
assert (df["Destino/Origen"].eq("Namibia") & df["codigo_pais"].eq("NA")).sum() == 173

# Registro de los resultados de carga por año.
pd.DataFrame(control).to_csv(
    CARPETA_SALIDA / "control_carga.csv", index=False, encoding="utf-8-sig"
)
print("Base inicial:", len(df), "registros")

In [ ]:
# 4. Depuración de datos y aplicación de correcciones.
# 4.1 Asignación del continente para los registros del Congo en 2024.
mascara = df["Año"].eq(2024) & df["codigo_pais"].eq("CG") & df["Continente"].isna()
assert mascara.sum() == 60

df.loc[mascara, "Continente"] = "África"

# 4.2 Corrección de las denominaciones de países según su código.
df.loc[df["codigo_pais"].eq("CD"), "Destino/Origen"] = "República Democrática del Congo"
df.loc[df["codigo_pais"].eq("CG"), "Destino/Origen"] = "República del Congo"
mascara_niger = df["codigo_pais"].eq("NE") & df["Destino/Origen"].eq("Nigeria")
assert mascara_niger.sum() == 5
df.loc[mascara_niger, "Destino/Origen"] = "Níger"

# 4.3 Homologación de los registros sin destino identificado.
mascara_x = (df["codigo_pais"].eq("X") &
             df["Destino/Origen"].eq("Sin Descripcion Segun SUNAT"))
assert mascara_x.sum() == 2
df.loc[mascara_x, "codigo_pais"] = "99"

# 4.4 Recuperación del sector mediante la subpartida arancelaria.
sin_sector = df["Sector"].isna()
assert sin_sector.sum() == 46 and df.loc[sin_sector, "Año"].eq(2019).all()
ref = df.loc[df["Año"].eq(2019) & df["Sector"].notna(),
             ["Subpartida", "Sector"]].drop_duplicates()
conteo = ref.groupby("Subpartida")["Sector"].nunique()
subpartidas_sin_sector = df.loc[sin_sector, "Subpartida"].unique()
assert conteo.reindex(subpartidas_sin_sector).eq(1).all()
mapa = ref.drop_duplicates("Subpartida").set_index("Subpartida")["Sector"]
df.loc[sin_sector, "Sector"] = df.loc[sin_sector, "Subpartida"].map(mapa)

# 4.5 Identificación y exclusión de subpartidas con códigos que comienzan en 000.
mascara_000 = df["Subpartida"].str.startswith("000").fillna(False)
assert mascara_000.sum() == 13
excluidos_000 = df.loc[mascara_000].copy()
excluidos_000.to_csv(
    CARPETA_SALIDA / "subpartidas_excluidas_000.csv", index=False, encoding="utf-8-sig"
)
df = df.loc[~mascara_000].copy()

# 4.6 Eliminación de un registro duplicado tras revisar la fuente original.
assert df.duplicated().sum() == 1
df = df.drop_duplicates(keep="first").copy()

# 4.7 Validación de la base depurada
assert df.shape == (1_988_937, 17)
assert df["Subpartida"].nunique() == 7_364
assert df["codigo_pais"].isna().sum() == 0  # Incluye 'NA' para Namibia.
assert df["Sector"].isna().sum() == 0
assert df["codigo_via"].isna().sum() == 46
assert df["Vía de Transporte"].isna().sum() == 46
assert df.duplicated().sum() == 0
assert np.isfinite(df[["US$ millones", "TM"]].to_numpy(dtype=float)).all()
assert (df[["US$ millones", "TM"]] >= 0).all().all()

In [ ]:
# 5. Unificación de la clasificación sectorial con base en el año 2025.
sector_por_codigo = df.groupby("Subpartida")["Sector"].nunique()
codigos_revisar = sector_por_codigo[sector_por_codigo > 1].index
assert len(codigos_revisar) == 103

# Verificación del sector asignado a estas subpartidas en 2025.
referencia_2025 = (df.loc[df["Año"].eq(2025) & df["Subpartida"].isin(codigos_revisar)]
                   .groupby("Subpartida")["Sector"])
assert referencia_2025.nunique().eq(1).all()

sector_2025 = referencia_2025.first()
assert len(sector_2025) == 103 and sector_2025.eq("Papel y cartón").all()

# Aplicación de la clasificación de referencia a todo el periodo.
mascara_sector = df["Subpartida"].isin(codigos_revisar)
df.loc[mascara_sector, "Sector"] = (
    df.loc[mascara_sector, "Subpartida"].map(sector_2025).to_numpy()
)

In [ ]:
# 6. Validación final y almacenamiento de la base consolidada.
assert df.shape == (1_988_937, 17)
assert df["Subpartida"].nunique() == 7_364
assert df.groupby("Subpartida")["Sector"].nunique().eq(1).all()
assert df.duplicated().sum() == 0

# Guardado de la base con la clasificación sectorial homologada.
carpeta_procesados = Path("datos/procesados")
carpeta_procesados.mkdir(parents=True, exist_ok=True)
archivo_final = carpeta_procesados / "exportaciones_2011_2025_base_depurada.parquet"
df.to_parquet(archivo_final, index=False)

# Resumen de resultados
print("Base final:", len(df), "registros;", df["Subpartida"].nunique(), "subpartidas")
print("FOB acumulado (millones US$):", df["US$ millones"].sum())
print("Archivo guardado en:", archivo_final.resolve())